Imports

In [1]:
from qiskit_metal import designs
from qiskit_metal import MetalGUI, Dict, open_docs
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
import pyEPR as epr
import numpy as np
from qiskit_metal.renderers.renderer_ansys.hfss_renderer import QHFSSRenderer
from qiskit_metal.draw.utility import to_vec3D
from pyEPR.ansys import parse_units

# Closes all previous GUI instances when running the program again
try:
    gui.main_window.force_close = True
    gui.main_window.close()
except (NameError, AttributeError):
    pass


Universal Parameters

In [2]:
main_x = '5mm'
main_y = '5mm'
main_z = '-380um'
radbound_top='2mm'
radbound_bottom='380um'

lp_width = '180um'
lp_gap = '140um'

tl_width = '15.5um'
tl_gap = '9um'

rr_width = tl_width
rr_gap = tl_gap
rr_coupling_gap = '15um' 
rr_coupling_length = '850um'
rr_termination = rr_gap

## Resonator 3: 3mm (driven modal port test)

In [3]:
#design and GUI intialization
R3 = designs.DesignPlanar()
R3.overwrite_enabled = True
gui = MetalGUI(R3)

#chip & sample holder dimensions
R3.chips.main.size.size_x = main_x
R3.chips.main.size.size_y = main_y
R3.chips.main.size.size_z = main_z
R3.chips.main.size.sample_holder_top = radbound_top
R3.chips.main.size.sample_holder_bottom = radbound_bottom

#launch pad definition
R3_LP1 = LaunchpadWirebondDriven(R3,name='LP1',
                       options={'orientation': '0', 'pos_x': '-2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap ,'pad_height':'180um', 'trace_width': tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

R3_LP2 = LaunchpadWirebondDriven(R3,name='LP2',
                       options={'orientation': '180', 'pos_x': '2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap, 'pad_height':'180um', 'trace_width':tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

#resonator definition
R3_OTG = OpenToGround(R3,name='OTG'
                 ,options={'pos_x':'-0.425mm','pos_y':'-1.0mm','orientation':'-90','width':rr_width, 'gap':rr_gap, 'termination_gap': rr_termination })


R3_CLT = CoupledLineTee(R3,'CLT', options=dict(pos_x = '0mm', pos_y = '0mm',
                                                        prime_width=tl_width,
                                                        prime_gap=tl_gap,
                                                        second_width=rr_width,
                                                        second_gap=rr_gap,
                                                        fillet='50um',
                                                        mirror=True,
                                                        orientation = '0',
                                                        coupling_space = rr_coupling_gap,                                                         
                                                        coupling_length = rr_coupling_length,
                                                        open_termination = False))

RR3 = RouteMeander(R3, 'RR3',  Dict(
        trace_width =tl_width,
        trace_gap =tl_gap,
        total_length='2150um',
        hfss_wire_bonds = True,
        fillet='50 um',
        lead = dict(start_straight='100um',end_straight='10um'),
        meander= dict(spacing='150um',asymmetry='200um'),
        pin_inputs=Dict(
            start_pin=Dict(component='CLT', pin='second_end'),
            end_pin=Dict(component='OTG', pin='open')), ))

#Tx line segments

seg1 = RoutePathfinder(R3, 'seg1', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='LP1',
                                                                pin='tie'),
                                                       end_pin=dict(
                                                        component='CLT',
                                                        pin='prime_start')
                                                 )))


seg2 = RoutePathfinder(R3, 'seg2', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='CLT',
                                                                pin='prime_end'),
                                                       end_pin=dict(
                                                        component='LP2',
                                                        pin='tie')
                                                 )))


gui.rebuild()
gui.autoscale()


10:53AM 53s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).


Driven modal render — creates lumped ports at LP1/LP2 during import

In [4]:
pinfo = epr.ProjectInfo(
    project_path=r'C:\Users\labuser\Documents\Ansoft\Thomas',
    project_name='impedance_study',
    design_name='check_Z0'
)

hfss = R3.renderers.hfss
hfss.start()
hfss.render_design()

INFO 10:53AM [connect_project]: Connecting to Ansys Desktop API...


Exception: ERROR! Valid directory, but invalid project filename. 😭 Not found!                     Please check your filename.
C:\Users\labuser\Documents\Ansoft\Thomas\impedance_study.aedt
